<a href="https://colab.research.google.com/github/byu-cs-452/byu-cs-452-class-content/blob/main/mongo/Step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3 — Aggregation Pipelines: `$unwind`, `$lookup`, `$project`, and `$group`

In this activity we'll work through two guided examples that combine multiple aggregation stages.
By the end you should be comfortable with:

| Stage | What it does |
|---|---|
| `$unwind` | Flatten an array field into one doc per element |
| `$lookup` | Left-outer-join another collection |
| `$project` | Reshape documents — rename, compute, include/exclude fields |
| `$group` | Aggregate values (sum, avg, count, first, etc.) |

These are exactly the tools you'll need for the Step 5 COVID homework.

> **FYI:** There's also a `$out` stage that writes pipeline results to a **new collection** — like `CREATE TABLE AS SELECT` in SQL. We won't use it here since we're on a shared read-only connection, but you'll want it when working on your own Atlas cluster.

---
## Setup

In [ ]:
!pip install pymongo

In [ ]:
import pymongo

user = "class"
password = "184vLpDKvOhvv528"
cluster = "cluster0"
dnsprefix = "wvdjn"
connectionUrl = f"mongodb+srv://{user}:{password}@{cluster}.{dnsprefix}.mongodb.net/"
client = pymongo.MongoClient(connectionUrl)
print(f"Ping result: {client.admin.command('ping')}")

db = client.movies
client.admin.command('ping')
print("Connected!")

---
## Example 1 — `$unwind` + `$group`: Count actors per genre

**Goal:** For each genre, count how many actor appearances there are and find the average cast size.

Each movie document has an embedded `actors` array. `$unwind` explodes that array
so we get **one document per actor appearance**, then we group by genre.

In [ ]:
pipeline = [
    # Step 1: Unwind the embedded actors array — one doc per actor per movie
    {"$unwind": "$actors"},
    # Step 2: Group by genre, count actor appearances and distinct movies
    {"$group": {
        "_id": "$genre",
        "total_actor_appearances": {"$sum": 1},
        "movies_with_actors":      {"$addToSet": "$title"}
    }},
    # Step 3: Project to compute average cast size
    {"$project": {
        "_id": 0,
        "genre": "$_id",
        "total_actor_appearances": 1,
        "movie_count": {"$size": "$movies_with_actors"},
        "avg_cast_size": {
            "$round": [{"$divide": ["$total_actor_appearances", {"$size": "$movies_with_actors"}]}, 1]
        }
    }},
    {"$sort": {"total_actor_appearances": -1}}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

### Key takeaways
- `$unwind` turns an **embedded array** into separate documents — like normalizing on the fly.
- `$addToSet` collects unique values during a `$group` (we used it to count distinct movies).
- `$project` can compute new fields with expressions like `$divide`, `$size`, and `$round`.
- Stages execute **in order** — a pipeline is just a list of transformations.

---
## Example 2 — `$lookup` + `$unwind` + `$project`: Flatten movies with ratings

**Goal:** Produce a clean, flat view of each movie with its `title`, `genre`, `country`, and `rating`.

The `ratings` collection is a separate collection that stores `movie_id` and `rating`.
We need `$lookup` to join it onto `movies` — this is the MongoDB equivalent of a SQL `SELECT ... JOIN ... ON ...`.

In [ ]:
pipeline = [
    # Join ratings onto movies (ratings.movie_id matches movies._id)
    {"$lookup": {
        "from": "ratings",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "rating_info"
    }},
    # preserveNullAndEmptyArrays = True keeps movies even if no rating (LEFT JOIN)
    {"$unwind": {"path": "$rating_info", "preserveNullAndEmptyArrays": True}},
    # Clean output — only the fields we want, with a fallback for missing ratings
    {"$project": {
        "_id": 0,
        "title": 1,
        "genre": 1,
        "country": 1,
        "rating": {"$ifNull": ["$rating_info.rating", "N/A"]}
    }}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

### Key takeaways
- `$lookup` joins a **separate collection** by matching field values — like a SQL JOIN.
- `preserveNullAndEmptyArrays: True` makes `$unwind` behave like a **LEFT JOIN** instead of an INNER JOIN.
- `$ifNull` provides a default when a field is missing — handy for keeping output clean.
- This "join → unwind → project" pattern is one you'll use constantly.

---
## Now you try!

Use the `movies` and `ratings` collections for the exercises below.

### Exercise 1

Find the **top-rated movie per genre**.

Hints:
- `$lookup` ratings onto movies
- `$unwind` the ratings
- `$project` to keep `title`, `genre`, and `rating`
- `$sort` by rating descending, then `$group` by genre with `$first`

In [ ]:
# your code here

### Exercise 2

For each genre, compute:
- `movie_count` — number of movies
- `avg_rating` — average rating
- `avg_cast_size` — average number of actors

Sort by `avg_rating` descending.

Hints:
- `$lookup` ratings onto movies
- `$unwind` the ratings array
- `$project` to pull out `rating` and compute `cast_size` using `$size` on the `actors` array
- `$group` by genre with `$avg` and `$sum`
- Use `$project` at the end to rename `_id` to `genre` and round the averages

In [ ]:
# your code here